In [50]:
import sys
import os
import glob
import importlib

import tqdm

In [51]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as pp

%matplotlib inline

In [52]:
import jax
jax.config.update('jax_enable_x64', True)

import jax.random
import jax.numpy as jnp

In [53]:
import discovery as ds
import discovery.models.nanograv as ds_nanograv
import discovery.samplers.numpyro as ds_numpyro
importlib.reload(ds)
importlib.reload(ds_nanograv)
importlib.reload(ds_numpyro)
importlib.reload(ds.pulsar)
importlib.reload(ds.likelihood)
importlib.reload(ds.signals)
importlib.reload(ds.matrix)

<module 'discovery.matrix' from '/h/mit227/discovery-ppa/src/discovery/matrix.py'>

In [54]:
psr = ds.Pulsar.read_feather('/DATA/RUBICON_1/joh414/Jackson/subband_PA_data/J0835-4510.PA.head_derot.feather', json_metadata=False)
print(psr)

<Pulsar J0835-4510: 977 res, 0 pars>


In [55]:
with np.printoptions(threshold=sys.maxsize):
    print(psr.residuals)
    print(psr.toaerrs)

[ 1.39990696  1.23355418  1.11609147  1.04155355  0.9782128   0.86806798
  0.78404612  0.68926173  1.42411891  1.2644213   1.13727108  1.0601281
  0.99701383  0.89166793  0.81355935  0.7261348   1.39805139  1.24355237
  1.13190787  1.05254947  0.98565113  0.87822396  0.80032666  0.71355865
  1.42104523  1.24276122  1.12238174  1.04255151  0.97663322  0.8680924
  0.78774968  0.69188466  1.44038359  1.25987815  1.13399571  1.05478082
  0.98576047  0.88245545  0.76698053  0.68710044  1.45692645  1.28636809
  1.152699    1.06916244  1.00239758  0.89845356  0.81624934  0.72583195
  1.47466609  1.29403686  1.15229729  1.06449384  0.99415795  0.88671857
  0.80153725  0.70757564  1.3117951   1.07108538  0.99693017  0.87932607
  0.79282227  0.69412576  1.52806475  1.31208133  1.16518191  1.07412696
  1.00403003  0.89126993  0.80961669  0.71806674  1.53668121  1.31700612
  1.16779407  1.07866622  1.00656442  0.89492406  0.81272025  0.72232216
  1.50732892  1.30684585  1.1595157   1.07156552  1.0

In [56]:
print(str(b'J0835-4510'))

b'J0835-4510'


In [57]:
type(psr.residuals)

numpy.ndarray

#### Measurement noise only, no backends

In [58]:
m = ds.PulsarLikelihood([psr.residuals,#np.nan_to_num(psr.residuals, nan=0.0),
                         ds.makenoise_measurement_simple(psr)])

In [59]:
m.logL.params

['J0835-4510_efac', 'J0835-4510_log10_t2equad']

In [60]:
p0 = ds.sample_uniform(m.logL.params)
print(p0)
print(m.logL(p0))

{'J0835-4510_efac': 5.992138784628185, 'J0835-4510_log10_t2equad': -7.729994674567118}
-4194790519.476492


#### Add ECORR noise (GP), free params, no backends

In [61]:
m_ecorr = ds.PulsarLikelihood([psr.residuals,
                         ds.makenoise_measurement_simple(psr),
                         ds.makegp_ecorr_simple(psr)])
print(m_ecorr.logL.params)

['J0835-4510_efac', 'J0835-4510_log10_ecorr', 'J0835-4510_log10_t2equad']


In [62]:
p0 = ds.sample_uniform(m_ecorr.logL.params)
print(p0)
print(f'w/  ECORR: {m_ecorr.logL(p0)}')
print(f'w/o ECORR: {m.logL(p0)}')

{'J0835-4510_efac': 5.541861294195188, 'J0835-4510_log10_ecorr': -7.324757554487506, 'J0835-4510_log10_t2equad': -8.055697199192414}
w/  ECORR: -4904139347.412703
w/o ECORR: -4904139997.283346


#### Add RM noise

In [ ]:
m_RM = ds.PulsarLikelihood([psr.residuals,
                         ds.makenoise_measurement_simple(psr),
                         ds.makegp_ecorr_simple(psr),
                         ds.makegp_fourier(psr, ds.powerlaw, components=30, fourierbasis=ds.fourierbasis_dm)
                         ])
print(m_RM.logL.params)

['J0835-4510_efac', 'J0835-4510_fourierGP_gamma', 'J0835-4510_fourierGP_log10_A', 'J0835-4510_log10_ecorr', 'J0835-4510_log10_t2equad']


In [66]:
print(m_RM.logL(p0))

{'J0835-4510_efac': 5.541861294195188, 'J0835-4510_log10_ecorr': -7.324757554487506, 'J0835-4510_log10_t2equad': -8.055697199192414}


KeyError: 'J0835-4510_fourierGP_log10_A'